# wandb-config-into-args — faded example 2: Two-tier override: sweep config then CLI flags

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-config-into-args`. Running the beacon reports progress on the `Logging: wandb.config into args` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.config into args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-config-into-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-config-into-args"
DD_SUBTOPIC = "Logging: wandb.config into args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In a sweep with CLI override support, sweep-sampled values are applied first, then command-line flags override on top. The same `hasattr`/`setattr` loop runs twice — once for `sweep_cfg`, once for `cli_overrides` — with `cli_overrides` applied last so its values win.

## Faded exercise 2

Implement `apply_two_tier(args, sweep_cfg, cli_overrides)`. Apply `sweep_cfg` first (lower precedence), then `cli_overrides` (higher precedence). Use the `hasattr`/`setattr` pattern for both. Return `args`.

Complete the blanked step that applies the CLI overrides.

**Fill in:** Iterate cli_overrides.items() and apply each key-value pair to args using the hasattr/setattr guard, mirroring the sweep_cfg loop.

In [ ]:
from dataclasses import dataclass

@dataclass
class Args:
    lr: float = 1e-3
    batch_size: int = 64
    n_epochs: int = 5
    wandb_project: str = 'proj'

def apply_two_tier(args, sweep_cfg, cli_overrides):
    # Tier 1: sweep config (lower precedence)
    for k, v in sweep_cfg.items():
        if hasattr(args, k):
            setattr(args, k, v)
    # Tier 2: CLI overrides (higher precedence — applied after so they win)
    raise NotImplementedError()  # TODO: Iterate cli_overrides.items() and apply each key-value pair to args using the hasattr/setattr guard, mirroring the sweep_cfg loop.
    return args

args = Args()
apply_two_tier(args, {'lr': 3e-4, 'batch_size': 256}, {'lr': 1e-5})
print('lr (cli wins):', args.lr)           # 1e-5
print('batch_size (sweep):', args.batch_size)  # 256


from dataclasses import dataclass

@dataclass
class Args:
    lr: float = 1e-3
    batch_size: int = 64
    n_epochs: int = 5
    wandb_project: str = 'proj'

def apply_two_tier(args, sweep_cfg, cli_overrides):
    for k, v in sweep_cfg.items():
        if hasattr(args, k):
            setattr(args, k, v)
    for k, v in cli_overrides.items():
        if hasattr(args, k):
            setattr(args, k, v)
    return args

def _test():
    args = Args()
    result = apply_two_tier(
        args,
        {'lr': 3e-4, 'batch_size': 256, '_wandb': 1},
        {'lr': 1e-5, 'nonexistent': 99}
    )
    assert result is args
    assert result.lr == 1e-5, 'CLI override must win'
    assert result.batch_size == 256, 'sweep value must stick when no CLI override'
    assert result.n_epochs == 5, 'unsampled field unchanged'
    assert not hasattr(args, 'nonexistent')


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass

@dataclass
class Args:
    lr: float = 1e-3
    batch_size: int = 64
    n_epochs: int = 5
    wandb_project: str = 'proj'

def apply_two_tier(args, sweep_cfg, cli_overrides):
    for k, v in sweep_cfg.items():
        if hasattr(args, k):
            setattr(args, k, v)
    for k, v in cli_overrides.items():
        if hasattr(args, k):
            setattr(args, k, v)
    return args

args = Args()
apply_two_tier(args, {'lr': 3e-4, 'batch_size': 256}, {'lr': 1e-5})
print('lr (cli wins):', args.lr)
print('batch_size (sweep):', args.batch_size)
```
</details>